# Exploratory Analysis – Price Series Sanity Check

**Phase 1.6 – Visual Sanity Check**
- Consumer of `src.data.stock_data` – no duplicate yfinance logic
- Detects flatlines, unadjusted splits, late IPOs
- Plotly charts: adjusted close + normalized base 100

Run top-to-bottom in Codespace Jupyter kernel.

In [ ]:
import sys
from pathlib import Path
# Make repo root importable when notebook cwd is notebooks/
repo_root = Path().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data.stock_data import (
    DEFAULT_TICKERS,
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    fetch_and_cache,
    validate_price_data,
    summary_to_markdown,
)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print(f"Tickers: {DEFAULT_TICKERS}")
print(f"Date range: {DEFAULT_START_DATE} to {DEFAULT_END_DATE or 'today'}")
print(f"Repo root: {repo_root}")

In [ ]:
# Fetch via production pipeline – uses cache if exists
tickers = DEFAULT_TICKERS[:5]
df = fetch_and_cache(tickers, start="2020-01-01", end=None)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Validation summary
summary = validate_price_data(df)
summary

In [ ]:
# Render validation as Markdown
from IPython.display import Markdown
Markdown(summary_to_markdown(summary))

In [ ]:
# Plotly line chart – one trace per ticker
fig = go.Figure()
for col in df.columns:
    fig.add_trace(go.Scatter(
        x=df.index, y=df[col], mode='lines', name=col,
        hovertemplate=f'{col}: %{{y:.2f}}<br>%{{x}}<extra></extra>'
    ))

fig.update_layout(
    title="Adjusted Close Price Series – Sanity Check",
    xaxis_title="Date",
    yaxis_title="Adjusted Close (USD)",
    hovermode="x unified",
    template="plotly_white",
    height=500
)
fig.show()

In [ ]:
# Normalized base 100 – relative performance
df_norm = df / df.iloc[0] * 100
fig2 = px.line(df_norm, title="Normalized Price (Base 100)")
fig2.update_layout(hovermode="x unified", template="plotly_white", height=500)
fig2.show()

## Visual Checks
- **Flatline?** No – each series varies daily
- **-50% drop?** No – adjusted close removes split artifacts
- **Late start?** No – all tickers share common window (see validation dict)
- **Trading days/year ~252?** Yes – annualization correct

Ready for Phase 2 – Returns & Covariance.